# Experimento — SOM — reconhecimento de caracteres 5×7

**Aula 2 · 18/11 · Inteligência Artificial aplicada à Engenharia Elétrica**

## Antes de começar

### Objetivo
Treinar uma grade SOM 5×5 para representar 10 dígitos binários de 35 pixels. Entradas e distâncias são adimensionais.

### Pré-requisitos
Nenhum notebook anterior. Execute este arquivo de forma independente em uma sessão nova do Google Colab.

### Como executar
1. Abra o link público deste experimento no Colab: [Abrir SOM — reconhecimento de caracteres 5×7](https://colab.research.google.com/github/Furriel/code-library/blob/main/cursos/ia/redes-neurais/som-reconhecimento.ipynb).
2. No menu **Ambiente de execução → Executar tudo**, rode as células de cima para baixo.
3. Faça a conta à mão antes de verificar a célula numérica.
4. Altere **somente o parâmetro sinalizado** no experimento guiado e execute tudo novamente.
5. Compare os gráficos e registre as conclusões; você não precisa enviar o notebook nem utilizar GitHub.

### Dados e parâmetros
BMU, calibração, rótulos, erros com dois pixels invertidos, EQ e alterações controladas na taxa inicial.

O ambiente usa NumPy e Matplotlib disponíveis no Colab. O compilador C é opcional onde houver exemplo de referência.


In [ ]:
import json, math, shutil, subprocess
import numpy as np
import matplotlib.pyplot as plt

GCC = shutil.which("gcc") is not None
print("GCC disponível:", GCC)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

class LCG:
    # Mesmo gerador da página: s <- (1664525*s + 1013904223) mod 2^32; u = s / 2^32.
    def __init__(self, semente):
        self.s = semente % 2**32
    def proximo(self):
        self.s = (self.s * 1664525 + 1013904223) % 2**32
        return self.s / 4294967296
    def inteiro(self, n):
        return int(self.proximo() * n)

## 1. SOM de caracteres 5×7 — a mesma implementação do simulador

Cada dígito é uma matriz de 5 colunas × 7 linhas, lida por linhas como um vetor de 35 valores binários. A SOM tem 25 neurônios (grade 5×5), cada um com 35 pesos.

- **Treino (não supervisionado):** glifo sorteado e, com probabilidade 0,75, um pixel invertido. O nome do glifo não entra na atualização.
- **Esquemas:** $\alpha(t)=0{,}30\,e^{-t/800}+0{,}02$; $\sigma(t)=1{,}6\,e^{-t/600}+0{,}35$.
- **Critério de parada:** a cada 100 atualizações, $|\Delta EQ|\le 0{,}004$ e variação média dos pesos $\le 0{,}004$ por 3 verificações seguidas; limite de segurança de 20 000. Não usa rótulos nem o dígito escolhido.
- **Calibração (depois do treino):** 360 padrões rotulados (glifo limpo + 35 versões com um pixel invertido) votam em suas BMUs. Não altera pesos.
- **Teste:** 500 padrões com dois pixels invertidos, que não aparecem no treino nem na calibração.

As funções abaixo replicam `nucleo-som.js` (mesmo gerador, mesma ordem). A célula de verificação grava `som_notebook_resultado.json`, usado pelos testes automáticos para comparar com o navegador.

In [ ]:
FONTE = {
 "0": ["01110","10001","10011","10101","11001","10001","01110"], "1": ["00100","01100","00100","00100","00100","00100","01110"],
 "2": ["01110","10001","00001","00010","00100","01000","11111"], "3": ["11110","00001","00001","01110","00001","00001","11110"],
 "4": ["00010","00110","01010","10010","11111","00010","00010"], "5": ["11111","10000","10000","11110","00001","00001","11110"],
 "6": ["01110","10000","10000","11110","10001","10001","01110"], "7": ["11111","00001","00010","00100","01000","01000","01000"],
 "8": ["01110","10001","10001","01110","10001","10001","01110"], "9": ["01110","10001","10001","01111","00001","00001","01110"]}
DIGITOS = list(FONTE)
PADROES = {d: np.array([int(c) for c in "".join(FONTE[d])], dtype=float) for d in DIGITOS}
POS = np.array([(j // 5, j % 5) for j in range(25)], dtype=float)
CFG = dict(semente=20261104, alfa0=.30, alfaMin=.02, tauAlfa=800, sigma0=1.6, sigmaMin=.35, tauSigma=600,
           probRuido=.75, intervalo=100, tolEQ=.004, tolPesos=.004, paciencia=3, maxAtualizacoes=20000)

def inverter(x, i):
    y = x.copy(); y[i] = 1 - y[i]; return y
CAL_X = np.array([PADROES[d] if i < 0 else inverter(PADROES[d], i) for d in DIGITOS for i in range(-1, 35)])
CAL_D = [d for d in DIGITOS for _ in range(36)]

def alfa(t, c=CFG): return c["alfa0"] * math.exp(-t / c["tauAlfa"]) + c["alfaMin"]
def sigma(t, c=CFG): return c["sigma0"] * math.exp(-t / c["tauSigma"]) + c["sigmaMin"]

def metricas(W, X=CAL_X):
    D = ((X[:, None, :] - W[None]) ** 2).sum(2)
    o = np.argsort(D, axis=1, kind="stable")
    adj = np.abs(POS[o[:, 0]] - POS[o[:, 1]]).sum(1) == 1
    return float(np.mean(np.sqrt(D[np.arange(len(X)), o[:, 0]]))), float(np.mean(~adj))

def treinar_som(cfg=CFG, registrar=()):
    rng = LCG(cfg["semente"])
    W = np.array([[0.08 + 0.84 * rng.proximo() for _ in range(35)] for _ in range(25)])
    t, estaveis, ant = 0, 0, W.copy()
    eq0, et0 = metricas(W); eq_ant = eq0
    verif, snaps = [(0, eq0, et0, None, None)], {0: W.copy()}
    status = "limite"
    while t < cfg["maxAtualizacoes"]:
        d = DIGITOS[rng.inteiro(10)]
        x = PADROES[d].copy()
        if rng.proximo() < cfg["probRuido"]:
            x = inverter(x, rng.inteiro(35))
        a, s = alfa(t, cfg), sigma(t, cfg)
        c = int(np.argmin(((x - W) ** 2).sum(1)))
        h = np.exp(-((POS - POS[c]) ** 2).sum(1) / (2 * s * s))
        W += a * h[:, None] * (x - W)
        t += 1
        if t in registrar: snaps[t] = W.copy()
        if t % cfg["intervalo"] == 0:
            eq, et = metricas(W)
            deq, dw = abs(eq - eq_ant), float(np.mean(np.abs(W - ant)))
            estavel = deq <= cfg["tolEQ"] and dw <= cfg["tolPesos"]
            estaveis = estaveis + 1 if estavel else 0
            eq_ant, ant = eq, W.copy()
            verif.append((t, eq, et, deq, dw))
            if estaveis >= cfg["paciencia"]:
                status = "convergiu"; break
    snaps[t] = W.copy()
    return W, t, status, verif, snaps

def calibrar(W):
    votos = np.zeros((25, 10), dtype=int)
    for x, d in zip(CAL_X, CAL_D):
        votos[int(np.argmin(((x - W) ** 2).sum(1))), DIGITOS.index(d)] += 1
    rot = [DIGITOS[int(np.argmax(v))] if v.max() > 0 else None for v in votos]
    return rot, votos

## Cálculo manual — distância da BMU

Um glifo 5×7 tem 35 elementos. Para um glifo limpo e sua cópia com **um** pixel invertido, a distância euclidiana ao protótipo limpo é `sqrt(1)=1` (os outros 34 pixels são iguais). Para duas diferenças, a distância é `sqrt(2)`; calcule também os dois quadrados de distância.


In [ ]:
x=PADROES["0"]; alterado=inverter(x,0)
d2=float(np.sum((x-alterado)**2))
assert len(x)==35 and d2==1
print("CÁLCULO MANUAL CONFERIDO: 1 pixel invertido; distância²=1; distância=1")


## 2. Treinar até o critério: evolução do erro e dos pesos

In [ ]:
W, t_final, status, verif, snaps = treinar_som(registrar={1, 10, 100, 300, 1000})
print(f"Status: {status} em {t_final} atualizações (limite {CFG['maxAtualizacoes']}).")
assert status == "convergiu", "Critério não atingido: analise a falha antes de interpretar o mapa."
with open("som_notebook_resultado.json", "w", encoding="utf-8") as arquivo:
    json.dump({"atualizacoes": t_final, "pesos": W.tolist()}, arquivo)

v = np.array([(a, b, c, np.nan if d is None else d, np.nan if e is None else e) for a, b, c, d, e in verif], dtype=float)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(v[:, 0], v[:, 1], label="erro de quantização EQ", color="#146e83")
ax[0].plot(v[:, 0], v[:, 2], ls="--", label="erro topográfico (fração)", color="#b45309")
ax[0].set(title="Qualidade do mapa", xlabel="atualizações", ylabel="EQ (distância em 35 dim.) · ET (fração)"); ax[0].legend()
ax[1].semilogy(v[1:, 0], v[1:, 3], label="|ΔEQ| entre verificações", color="#146e83")
ax[1].semilogy(v[1:, 0], v[1:, 4], label="variação média dos pesos", color="#7c3aed")
ax[1].axhline(CFG["tolEQ"], ls=":", color="#b42318", label="tolerância")
ax[1].set(title="Critério de parada", xlabel="atualizações", ylabel="variação (escala log)"); ax[1].legend()
plt.tight_layout(); plt.show()

def desenhar_mapa(ax, W, titulo, rotulos=None):
    img = np.ones((5 * 8, 5 * 6))
    for j in range(25):
        r, c = divmod(j, 5)
        img[r * 8:r * 8 + 7, c * 6:c * 6 + 5] = 1 - W[j].reshape(7, 5)
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    if rotulos:
        for j, rot in enumerate(rotulos):
            r, c = divmod(j, 5)
            ax.text(c * 6 + 2, r * 8 + 7.6, rot if rot else "—", ha="center", fontsize=8, color="#b45309")
    ax.set(title=titulo, xticks=[], yticks=[])

fig, ax = plt.subplots(1, len(snaps), figsize=(3.2 * len(snaps), 3.6))
for a, (t, Wt) in zip(ax, sorted(snaps.items())):
    desenhar_mapa(a, Wt, f"t = {t}")
fig.suptitle("Os 25 vetores de pesos (escuro = peso próximo de 1): do aleatório aos padrões")
plt.tight_layout(); plt.show()

## 3. Calibração, ocupação das unidades e mapa de distâncias

- **Rótulos:** associados depois do treino, por votação; unidades sem votos ficam sem rótulo.
- **Ocupação:** quantos padrões de calibração escolhem cada unidade como BMU.
- **Matriz U:** distância média entre os pesos de cada unidade e os de suas vizinhas na grade; valores altos marcam fronteiras entre grupos.

In [ ]:
rotulos, votos = calibrar(W)
ocup = votos.sum(1).reshape(5, 5)
U = np.zeros((5, 5))
for j in range(25):
    viz = [k for k in range(25) if np.abs(POS[k] - POS[j]).sum() == 1]
    U[j // 5, j % 5] = np.mean([np.linalg.norm(W[j] - W[k]) for k in viz])
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
desenhar_mapa(ax[0], W, f"Pesos finais e rótulos (t = {t_final})", rotulos)
im = ax[1].imshow(ocup, cmap="Blues"); fig.colorbar(im, ax=ax[1], shrink=.8, label="nº de padrões de calibração")
for j in range(25): ax[1].text(j % 5, j // 5, f"{ocup[j // 5, j % 5]}\nrót. {rotulos[j] or '—'}", ha="center", va="center", fontsize=9, color="white" if ocup[j // 5, j % 5] > 18 else "black")
ax[1].set(title="Ocupação das unidades (BMU)", xlabel="coluna da grade", ylabel="linha da grade")
im = ax[2].imshow(U, cmap="magma_r"); fig.colorbar(im, ax=ax[2], shrink=.8, label="distância média às vizinhas (35 dim.)")
ax[2].set(title="Matriz U (fronteiras entre grupos)", xlabel="coluna da grade", ylabel="linha da grade")
plt.tight_layout(); plt.show()
print("Unidades rotuladas:", sum(r is not None for r in rotulos), "de 25")

## 4. Reconhecimento dos dez dígitos, ruído e falhas

O reconhecimento compara a entrada com **todos** os 25 neurônios, escolhe a menor distância (BMU) e só então consulta o rótulo. Os pesos não mudam.

In [ ]:
def reconhecer(W, x):
    d2 = ((x - W) ** 2).sum(1)
    o = np.argsort(d2, kind="stable")
    return int(o[0]), rotulos[int(o[0])], float(np.sqrt(d2[o[0]])), float(np.sqrt(d2[o[1]]) - np.sqrt(d2[o[0]])), d2

W_antes = W.copy()
print("Glifos limpos:")
for d in DIGITOS:
    c, rot, dist, margem, _ = reconhecer(W, PADROES[d])
    print(f"  {d} -> BMU N{c + 1:02d}, rótulo {rot}, d = {dist:.3f}, margem = {margem:.3f}")
    assert rot == d

rng_t = LCG(7)                      # mesmo conjunto de teste da página
teste = []
for d in DIGITOS:
    for _ in range(50):
        i = rng_t.inteiro(35); j = rng_t.inteiro(34)
        if j >= i: j += 1
        teste.append((inverter(inverter(PADROES[d], i), j), d, (i, j)))
saidas = [reconhecer(W, x)[1] for x, _, _ in teste]
acertos = sum(s == d for s, (_, d, _) in zip(saidas, teste))
print(f"\nTeste com 2 pixels invertidos: {acertos}/{len(teste)} = {100 * acertos / len(teste):.1f} % (base sintética, não OCR real)")
assert np.array_equal(W, W_antes), "o reconhecimento não pode alterar pesos"

conf = np.zeros((10, 11), dtype=int)
for s, (_, d, _) in zip(saidas, teste):
    conf[DIGITOS.index(d), 10 if s is None else DIGITOS.index(s)] += 1
falhas = [(x, d, px, s) for s, (x, d, px) in zip(saidas, teste) if s != d]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.2, 1]})
im = ax[0].imshow(conf, cmap="Blues")
ax[0].set(title="Matriz de confusão (teste com 2 pixels invertidos)", xlabel="saída (— = sem rótulo)", ylabel="glifo de origem",
          xticks=range(11), xticklabels=DIGITOS + ["—"], yticks=range(10), yticklabels=DIGITOS)
for i in range(10):
    for j in range(11):
        if conf[i, j]: ax[0].text(j, i, conf[i, j], ha="center", va="center", color="white" if conf[i, j] > 25 else "black")
ax[1].axis("off"); ax[1].set_title(f"Primeiras falhas ({len(falhas)} no total)")
for k, (x, d, px, s) in enumerate(falhas[:8]):
    a = fig.add_axes([0.58 + (k % 4) * 0.095, 0.45 - (k // 4) * 0.38, 0.08, 0.3])
    a.imshow(1 - x.reshape(7, 5), cmap="gray", vmin=0, vmax=1); a.set(xticks=[], yticks=[])
    a.set_title(f"{d}→{s or '—'}", fontsize=9)
plt.show()

**Como ler as falhas.** Uma falha não é “erro do código”: duas inversões podem aproximar um glifo de outro (por exemplo, 8 e 9 diferem em poucos pixels), ou levar a BMU a uma unidade sem rótulo. A distância não é probabilidade e a rede não rejeita entradas desconhecidas: uma matriz vazia também tem BMU.

## 5. Das redes deste laboratório aos modelos de linguagem

| | Perceptron | MLP | SOM | Transformer / LLM |
|---|---|---|---|---|
| Parâmetros | 3 | 17 | 875 | bilhões |
| Treino | regra do perceptron | gradiente (retropropagação) | competição + vizinhança, sem rótulos | gradiente para prever o próximo token |
| Inferência | um limiar | composição de não linearidades | vetor de referência mais próximo | atenção entre posições e geração token a token |

Todos ajustam parâmetros no treino e os mantêm fixos na inferência. A BMU é uma escolha por distância a referências fixas; a atenção é uma soma ponderada aprendida entre posições do contexto. Uma SOM não modela sequências nem evolui para um Transformer.

## 6. Experimento guiado C — SOM: alterar a taxa inicial e conferir reconhecimento

A SOM aprende **sem rótulos**. Depois do treino, a calibração associa rótulos aos neurônios; por último, testamos padrões sintéticos com dois pixels invertidos.

1. Preveja se alterar `ALFA0_ALTERADA` aumentará ou reduzirá o erro de quantização.
2. Execute novamente a SOM com a **mesma semente, padrões e critérios**, mudando apenas `alfa0`; o modelo original não é sobrescrito.
3. Compare os gráficos de EQ e os acertos no mesmo conjunto de 500 padrões.
4. Se uma execução atingir o limite sem convergir, **não** a apresente como convergente. A taxa de acerto não é OCR real.

**Não selecione uma rede por este conjunto de avaliação**: ele serve aqui apenas para ilustrar diferenças num experimento didático. Registre os valores observados no relatório único.

Não há entrega por técnica: registre as evidências no **relatório final único do Moodle**.


In [ ]:
ALFA0_ALTERADA = 0.15  # Experimente 0.15 ou 0.45; original = 0.30
assert 0 < ALFA0_ALTERADA <= 1
cfg_som_alt = {**CFG, "alfa0": ALFA0_ALTERADA}
W_alt, t_alt, status_alt, verif_alt, _ = treinar_som(cfg=cfg_som_alt)
rot_alt, _ = calibrar(W_alt)
def contar_acertos_som(pesos, labels, exemplos):
    corretos = 0
    for xx, dg, _ in exemplos:
        bmu = int(np.argmin(((xx - pesos) ** 2).sum(1)))
        corretos += labels[bmu] == dg
    return corretos
acertos_alt = contar_acertos_som(W_alt, rot_alt, teste)
eq_original, eq_alterado = metricas(W)[0], metricas(W_alt)[0]
print("ANTES × DEPOIS — SOM (mesmo conjunto sintético)")
print(f"Original: alfa0={CFG['alfa0']:.2f}; status={status}; atualizações={t_final}; EQ={eq_original:.3f}; acertos={acertos}/{len(teste)}")
print(f"Alterada: alfa0={ALFA0_ALTERADA:.2f}; status={status_alt}; atualizações={t_alt}; EQ={eq_alterado:.3f}; acertos={acertos_alt}/{len(teste)}")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot([x[0] for x in verif], [x[1] for x in verif], label=f"original α₀={CFG['alfa0']:.2f}")
ax.plot([x[0] for x in verif_alt], [x[1] for x in verif_alt], label=f"alterada α₀={ALFA0_ALTERADA:.2f}")
ax.set(xlabel="atualizações", ylabel="erro de quantização (35 dimensões)", title="SOM: comparação do aprendizado")
ax.legend(); plt.tight_layout(); plt.show()
print("\nINTERPRETAÇÃO AUTOMÁTICA:")
if status_alt != "convergiu":
    print("A execução alterada não atingiu o critério de estabilidade. NÃO declare convergência.")
elif eq_alterado < eq_original:
    print("A alteração reduziu o EQ final, que mede proximidade aos protótipos, não acurácia.")
else:
    print("O EQ final não diminuiu; uma taxa diferente pode alterar a trajetória do treinamento.")
if acertos_alt > acertos:
    print("A calibração/avaliação no conjunto sintético teve mais acertos.")
elif acertos_alt < acertos:
    print("A calibração/avaliação no conjunto sintético teve menos acertos.")
else:
    print("A quantidade de acertos foi igual nas duas redes.")
print("EQ e acertos são grandezas diferentes; nenhum deles autoriza afirmar desempenho em OCR real.")



## Questões de interpretação

Qual a diferença entre EQ e acertos? Uma unidade sem rótulo é falha do algoritmo? Registre um cálculo manual, mapa de unidades, matriz de confusão e comparação no relatório único do Moodle.

## Conclusão

Este experimento é independente. Use o mesmo conjunto de dados e as mesmas condições iniciais ao comparar parâmetros. **Experimentos propostos:** altere apenas o valor indicado no código e justifique o resultado observado, sem afirmar melhora quando as métricas não a demonstram. Entregue somente um **único relatório final** pelo Moodle, conforme as instruções da disciplina.
